# Etapa 2: Analítica Diagnóstica — ¿Por qué pasó?
Partimos de los hallazgos descriptivos (rutas que fallan, productos sin inventario) y buscamos causas.
Técnicas: **desglose (drill-down)**, **árbol de clasificación** y **segmentación para evitar conclusiones engañosas**.
**Nota:** los datos son sintéticos, solo para ilustrar la técnica.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

rng = np.random.default_rng(42)
pd.set_option("display.width", 150)

## 1. Datos de ejemplo (mismo escenario de la etapa descriptiva)

In [ ]:
pools = {"R1":[1,2,3],"R2":[2,3,4],"R3":[4,5,9],"R4":[5,6,7],
         "R5":[7,8,9],"R6":[8,10,11],"R7":[9,11,12],"R8":[10,12,1]}
region_de = {"R1":"Central","R2":"Central","R3":"Pacífico","R4":"Pacífico",
             "R5":"Caribe","R6":"Caribe","R7":"Norte","R8":"Norte"}
n = 4000
fechas = pd.to_datetime("2025-01-01") + pd.to_timedelta(rng.integers(0, 365, n), unit="D")
rutas = rng.choice(list(pools), n)
choferes = [f"Chofer_{rng.choice(pools[r]):02d}" for r in rutas]
camiones = [f"T-{r}{rng.choice(['a','b'])}" for r in rutas]
mes = fechas.month
# Causas "ocultas" que el análisis debe descubrir:
#  - Chofer_09 falla en cualquier ruta donde trabaja
#  - R7 falla en época lluviosa (jun-oct) sin importar el chofer
p_fallo = 0.08 + np.where(np.array(choferes) == "Chofer_09", 0.30, 0) \
              + np.where((rutas == "R7") & (mes >= 6) & (mes <= 10), 0.15, 0)
entregas = pd.DataFrame({"fecha": fechas, "mes": mes, "ruta": rutas,
                         "region": [region_de[r] for r in rutas],
                         "chofer": choferes, "camion": camiones,
                         "fallida": rng.random(n) < p_fallo})

productos = ["Arroz","Frijoles","Aceite","Bloqueador","Gaseosa","Detergente","Leche"]
filas = []
for sem in range(52):
    mes_s = int(min(12, sem // 4.35 + 1))
    for p in productos:
        est = {"Bloqueador": 2.2 if mes_s <= 3 else 0.8,
               "Gaseosa": 1.8 if mes_s == 12 else 1.0}.get(p, 1.0)
        demanda = rng.poisson(100 * est)
        stock = rng.poisson(120)
        filas.append((sem, mes_s, p, demanda, stock, demanda > stock))
inventario = pd.DataFrame(filas, columns=["semana","mes","producto","demanda","stock","sin_inventario"])
print("Tasa de fallo global:", round(100 * entregas.fallida.mean(), 1), "%")

## 2. Técnica 1: Desglose (drill-down) de las rutas que fallan
Pregunta: ¿son **distintos camiones/choferes** fallando en la misma ruta, o **un chofer** que falla en varias rutas?

In [ ]:
def tasa(df, col):
    return (100 * df.groupby(col).fallida.mean()).round(1).sort_values(ascending=False)

print("Por región:\n", tasa(entregas, "region"), "\n")
print("Por ruta:\n", tasa(entregas, "ruta"), "\n")
print("Por chofer:\n", tasa(entregas, "chofer"))

In [ ]:
# Cruce ruta x chofer: la clave del diagnóstico
cruce = entregas.pivot_table(index="chofer", columns="ruta", values="fallida", aggfunc="mean") * 100
fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(cruce.values, cmap="Reds", aspect="auto")
ax.set_xticks(range(len(cruce.columns))); ax.set_xticklabels(cruce.columns)
ax.set_yticks(range(len(cruce.index))); ax.set_yticklabels(cruce.index)
for i in range(cruce.shape[0]):
    for j in range(cruce.shape[1]):
        v = cruce.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=8)
plt.colorbar(im, label="% fallo"); ax.set_title("% de fallo: chofer x ruta")
plt.tight_layout(); plt.show()

**Lectura:** la fila de `Chofer_09` está alta en R3, R5 y R7 (el problema sigue al chofer).
R7 también es alta con otros choferes: ahí hay un factor propio de la ruta. Veamos si es estacional.

In [ ]:
r7 = entregas[entregas.ruta == "R7"]
otros = entregas[entregas.ruta != "R7"]
r7_sin9 = r7[r7.chofer != "Chofer_09"]
fig, ax = plt.subplots(figsize=(9, 4))
(100 * r7_sin9.groupby("mes").fallida.mean()).plot(ax=ax, marker="o", label="R7 (sin Chofer_09)", color="#c0392b")
(100 * otros[otros.chofer != "Chofer_09"].groupby("mes").fallida.mean()).plot(ax=ax, marker="o", label="Otras rutas (sin Chofer_09)", color="gray")
ax.set_ylabel("% fallo"); ax.set_xlabel("Mes"); ax.set_title("R7 falla en la época lluviosa, aun sin Chofer_09")
ax.legend(); plt.tight_layout(); plt.show()

## 3. Técnica 2: Árbol de clasificación para el quiebre de inventario
Hipótesis ingenua: "se acabó porque se compró poco". El árbol revisa más factores (producto, mes, stock disponible).

In [ ]:
X = pd.get_dummies(inventario[["producto","mes","stock"]], columns=["producto"], dtype=int)
y = inventario.sin_inventario.astype(int)
arbol = DecisionTreeClassifier(max_depth=3, min_samples_leaf=10, random_state=0).fit(X, y)

print(f"Exactitud en entrenamiento: {arbol.score(X, y):.2f}\n")
print(export_text(arbol, feature_names=list(X.columns)))

fig, ax = plt.subplots(figsize=(14, 6))
plot_tree(arbol, feature_names=list(X.columns), class_names=["Con stock","Sin stock"],
          filled=True, rounded=True, fontsize=8, ax=ax)
plt.show()

imp = pd.Series(arbol.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nImportancia de variables:\n", imp[imp > 0].round(3))

In [ ]:
# Verificamos con datos lo que sugiere el árbol: el bloqueador solo falta en temporada
bloq = inventario[inventario.producto == "Bloqueador"]
print("Bloqueador: % de semanas sin inventario por mes")
print((100 * bloq.groupby("mes").sin_inventario.mean()).round(0).to_string())
print("\nConclusión: comprar más bloqueador TODO el año sería un error; el pico es estacional (ene-mar).")

## 4. Técnica 3: Contexto antes de comparar (el peligro de promediar sin segmentar)
Un modelo ingenuo vería que los camiones regionales tienen mayor velocidad promedio y concluiría que "pueden hacer más entregas".

In [ ]:
k = 40
zona = np.array(["Regional"] * 20 + ["Urbano"] * 20)
camiones = pd.DataFrame({
    "camion": [f"C{i:02d}" for i in range(k)], "zona": zona,
    "vel_prom_kmh": np.where(zona == "Regional", rng.normal(62, 4, k), rng.normal(28, 3, k)),
    "km_por_entrega": np.where(zona == "Regional", rng.normal(38, 5, k), rng.normal(4, 1, k)),
})
camiones["horas_por_entrega"] = camiones.km_por_entrega / camiones.vel_prom_kmh + 0.4  # +0.4h de descarga
camiones["entregas_por_dia"] = (8 / camiones.horas_por_entrega).round(1)
camiones["costo_por_entrega"] = camiones.km_por_entrega * 0.55 + 6           # combustible + costo fijo

print("VISTA INGENUA (todos juntos): ranking por velocidad")
print(camiones.sort_values("vel_prom_kmh", ascending=False).head(5)[["camion","zona","vel_prom_kmh"]].round(1))

print("\nVISTA SEGMENTADA POR ZONA:")
res = camiones.groupby("zona")[["vel_prom_kmh","km_por_entrega","entregas_por_dia","costo_por_entrega"]].mean().round(1)
print(res)

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(13, 4))
for ax, col, t in zip(axs, ["vel_prom_kmh","entregas_por_dia","costo_por_entrega"],
                      ["Velocidad (km/h)", "Entregas por día", "Costo por entrega"]):
    res[col].plot.bar(ax=ax, color=["#e67e22", "#16a085"]); ax.set_title(t); ax.tick_params(axis="x", rotation=0)
plt.suptitle("Los regionales son más rápidos, pero hacen menos entregas y cada una cuesta más")
plt.tight_layout(); plt.show()

## 5. Conclusiones del diagnóstico
1. **Chofer_09** concentra fallos en varias rutas: acción sobre la persona (capacitación, revisión), no sobre las rutas.
2. **R7** falla en época lluviosa con cualquier chofer: acción sobre la ruta (planificación estacional, rutas alternas).
3. El quiebre de **bloqueador** es estacional: ajustar el plan de compra por temporada, no subir el inventario base.
4. Comparar camiones sin segmentar por zona llevaría a aumentar entregas regionales y **subir costos**.
Estas variables y criterios son los que después alimentarían un modelo predictivo con sentido de negocio.